# Merge and extraction of Mission Gate videos

In [ ]:
#!/usr/bin/env python3
"""
Merge multiple CSV files and summarize frame ranges by seq and cam_view.

Requirements:
- Python 3
- pandas (`pip install pandas`)
"""

import pandas as pd
import glob
import os

# -------------------- USER SETTINGS --------------------
INPUT_CSV_FOLDER = "../data/csv_logs"  # Folder containing multiple CSV files
OUTPUT_CSV_FILE = "../data/csv_logs/merged_summary.csv" # Output CSV file
FILTER_SEQ = None       # e.g., "seq1" or None for all
FILTER_CAM_VIEW = None  # e.g., "cam1" or None for all
# -------------------------------------------------------

# Step 1: Read all CSV files
all_files = glob.glob(os.path.join(INPUT_CSV_FOLDER, "*.csv"))
if not all_files:
    print("No CSV files found in folder.")
    exit(1)

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

# Step 2: Filter seq and cam_view if specified
if FILTER_SEQ:
    df = df[df["seq"] == FILTER_SEQ]
if FILTER_CAM_VIEW:
    df = df[df["cam_view"] == FILTER_CAM_VIEW]

# Step 3: Group by seq and cam_view
grouped = df.groupby(["seq", "cam_view"])

summary_rows = []

for (seq, cam_view), group in grouped:
    # Determine min and max frame_num
    start_frame = group["frame_num"].min()
    end_frame = group["frame_num"].max()
    
    # Get first URL (assuming all rows in group are same video)
    url = group["url"].iloc[0]
    
    # Transfer additional info (assuming consistent within group)
    gait_event = group["gait_event"].iloc[0] if "gait_event" in group else ""
    dataset = group["dataset"].iloc[0] if "dataset" in group else ""
    gait_pat = group["gait_pat"].iloc[0] if "gait_pat" in group else ""
    
    summary_rows.append({
        "seq": seq,
        "cam_view": cam_view,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "url": url,
        "gait_event": gait_event,
        "dataset": dataset,
        "gait_pat": gait_pat
    })

# Step 4: Save to new CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV_FILE, index=False)
print(f"Summary CSV saved to {OUTPUT_CSV_FILE}")


/var/folders/kw/_ylp95111cz279gmvprkwk4c0000gn/T/ipykernel_7737/390037290.py:27: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(f) for f in all_files]


Summary CSV saved to ../data/merged_summary.csv


# Integrated version

In [14]:
#!/usr/bin/env python3
"""
Parallel, resumable YouTube frame-based segment downloader
with checksum logging.

Requirements:
- Python 3.9+
- pandas
- yt-dlp
- ffmpeg
- Optional but recommended: deno
"""

import os
import subprocess
import hashlib
import pandas as pd
from yt_dlp import YoutubeDL
from concurrent.futures import ProcessPoolExecutor, as_completed

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
OUTPUT_CSV = "../data/csv_logs/merged_summary_enriched.csv"

TEMP_FOLDER = "../data/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/video_snippets"

MAX_WORKERS = 4        # adjust for your machine
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# ------------------------------------------------------
# Utilities
# ------------------------------------------------------

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def get_video_info(url):
    with YoutubeDL({
        "quiet": True,
        "skip_download": True,
        "remote_components": "ejs:github",
    }) as ydl:
        return ydl.extract_info(url, download=False)

def download_full_video(url, title):
    with YoutubeDL({
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(TEMP_FOLDER, f"{title}.%(ext)s"),
        "remote_components": "ejs:github",
        "noplaylist": True,
        "quiet": True,
    }) as ydl:
        ydl.download([url])

def cut_segment(input_file, start_ts, end_ts, output_file):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", input_file,
        "-ss", start_ts,
        "-to", end_ts,
        "-c:v", "libx264",
        "-c:a", "aac",
        output_file
    ], check=True)

# ------------------------------------------------------
# Worker
# ------------------------------------------------------

def process_row(idx, row):
    try:
        output_name = (
            f"{row.seq}_{row.cam_view}_"
            f"{row.gait_event}_{row.dataset}_{row.gait_pat}.mp4"
        )
        output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

        # Resume: skip if already processed
        if os.path.exists(output_path) and not pd.isna(row.get("checksum", None)):
            return idx, None

        info = get_video_info(row.url)
        title = info["title"]
        uploader = info.get("uploader", "")

        fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
        fps = max(fps_list) if fps_list else 30

        start_ts = frame_to_timestamp(row.start_frame, fps)
        end_ts = frame_to_timestamp(row.end_frame, fps)
        duration = round((row.end_frame - row.start_frame) / fps, 3)

        # Download full video
        download_full_video(row.url, title)
        input_video = os.path.join(TEMP_FOLDER, f"{title}.mp4")

        # Cut snippet
        cut_segment(input_video, start_ts, end_ts, output_path)

        # Cleanup temp
        if os.path.exists(input_video):
            os.remove(input_video)

        checksum = sha256_checksum(output_path)

        return idx, {
            "title": title,
            "uploader": uploader,
            "fps": fps,
            "start_time": start_ts,
            "end_time": end_ts,
            "duration": duration,
            "checksum": checksum
        }

    except Exception as e:
        return idx, {"error": str(e)}

# ------------------------------------------------------
# Main
# ------------------------------------------------------

df = pd.read_csv(INPUT_CSV)

# Ensure columns exist (resume-safe)
for col in [
    "title", "uploader", "fps",
    "start_time", "end_time", "duration", "checksum"
]:
    if col not in df.columns:
        df[col] = ""

tasks = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for idx, row in df.iterrows():
        tasks.append(executor.submit(process_row, idx, row))

    for future in as_completed(tasks):
        idx, result = future.result()

        if result is None:
            continue

        if "error" in result:
            print(f"Row {idx} failed: {result['error']}")
            continue

        for k, v in result.items():
            df.at[idx, k] = v

# Save enriched CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Finished. CSV written to {OUTPUT_CSV}")


Process SpawnProcess-3:
Process SpawnProcess-2:
Process SpawnProcess-1:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/marcbp/.pyenv/versions/3.11.3/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/marcbp/.pyenv/versions/3.11.3/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/marcbp/.pyenv/versions/3.11.3/lib/python3.11/concurrent/futures/process.py", line 244, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/marcbp/.pyenv/versions/3.11.3/lib/python3.11/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'process_row' on <module '__main__' (built-in)>
  File "/Users/marcbp/.pyenv/versions/3.11.3/lib/python3.11/multiproces

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

### Removes an extra heading if necessary and formats the csv

In [38]:
import pandas as pd
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
df_raw = pd.read_csv(INPUT_CSV)

df = df_raw["merged_summary"].str.split(";", expand=True)

df.columns = [
    "seq",
    "cam_view",
    "start_frame",
    "end_frame",
    "url",
    "gait_event",
    "dataset",
    "gait_pat",
]

# 🔧 Remove rows where start_frame is not numeric (e.g. header rows)
df = df[df["start_frame"].str.isnumeric()]

# Convert numeric columns
df["start_frame"] = df["start_frame"].astype(int)
df["end_frame"] = df["end_frame"].astype(int)

# Validate required columns
required = {"seq", "cam_view", "start_frame", "end_frame", "url"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Overwrite original file
df.to_csv(INPUT_CSV, index=False)


KeyError: 'merged_summary'

# Reformats the csv if necessary

In [ ]:
#adjust the csv
import pandas as pd

INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"

# Load CSV
df_raw = pd.read_csv(INPUT_CSV)

# Detect merged column
if len(df_raw.columns) == 1:
    merged_col = df_raw.columns[0]
    print(f"Detected single merged column: '{merged_col}'")

    # Split by semicolon (adjust if your CSV uses commas)
    df = df_raw[merged_col].str.split(";", expand=True)

    df.columns = [
        "seq",
        "cam_view",
        "start_frame",
        "end_frame",
        "url",
        "gait_event",
        "dataset",
        "gait_pat",
    ]
else:
    df = df_raw.copy()
    print("CSV already has multiple columns, no split needed.")

# 🔧 Remove rows where start_frame is not numeric
df = df[df["start_frame"].apply(lambda x: str(x).isnumeric())]

# Convert numeric columns
df["start_frame"] = df["start_frame"].astype(int)
df["end_frame"] = df["end_frame"].astype(int)

# Overwrite original CSV
df.to_csv(INPUT_CSV, index=False)
print(f"✅ Cleaned CSV saved to {INPUT_CSV}")
print(df.head())


Detected single merged column: 'seq;cam_view;start_frame;end_frame;url;gait_event;dataset;gait_pat'
✅ Cleaned CSV saved to ../data/csv_logs/merged_summary_test.csv
                         seq    cam_view  start_frame  end_frame   
0  cljan9b4p00043n6ligceanyp  right side         1757       2268  \
1  cljanb45y00083n6lmh1qhydd   left side         2532       2746   
2  cljawsyn6001o3n6l6z20teaj  right side            1        449   
3  cljawu5xd001s3n6lejw8p0uv   left side          551        879   

                                           url gait_event        dataset   
0  https://www.youtube.com/watch?v=B5hrxKe2nP8             Abnormal Gait  \
1  https://www.youtube.com/watch?v=B5hrxKe2nP8             Abnormal Gait   
2  https://www.youtube.com/watch?v=IV_IsstW-gA             Abnormal Gait   
3  https://www.youtube.com/watch?v=IV_IsstW-gA             Abnormal Gait   

     gait_pat  
0  parkinsons  
1  parkinsons  
2    abnormal  
3    abnormal  


## Runs all videos in the list

In [42]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader with incremental CSV enrichment

- Downloads video if missing
- Cuts QuickTime-compatible MP4 clips
- Extracts metadata from downloaded or existing videos
- Updates enriched CSV row by row, even if video exists
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
OUTPUT_CSV = "../data/csv_logs/merged_summary_enriched.csv"
TEMP_FOLDER = "../data/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/video_snippets"
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

def download_full_video(url):
    ydl_opts = {
        "format": "bv*/b",  # best video or best combined
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")
    if info.get("vcodec") == "none":
        raise RuntimeError("Audio-only stream — no video available")

    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return info, input_path

def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    if not os.path.exists(output_file):
        subprocess.run([
            "ffmpeg", "-y",
            "-i", input_file,
            "-ss", start_ts,
            "-to", end_ts,
            "-c:v", "libx264",
            "-c:a", "aac",
            output_file
        ], check=True)

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# -------------------- Main ---------------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    # Add enrichment columns if missing
    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    for idx, row in df.iterrows():
        try:
            print(f"\n▶ Processing row {idx}")

            if pd.isna(row.url) or pd.isna(row.start_frame) or pd.isna(row.end_frame):
                print("Skipping row due to missing URL or frames")
                continue

            start_frame = int(row.start_frame)
            end_frame = int(row.end_frame)

            # Build output filename
            output_name = safe_name(f"{row.seq}_{row.cam_view}_{row.gait_event}_{row.dataset}_{row.gait_pat}.mp4")
            output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

            # Download video only if missing
            input_video = None
            if not os.path.exists(output_path):
                info, input_video = download_full_video(row.url)
                # Cut clip
                fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
                fps = max(fps_list) if fps_list else 30
                start_ts = frame_to_timestamp(start_frame, fps)
                end_ts = frame_to_timestamp(end_frame, fps)
                cut_and_reencode(input_video, start_ts, end_ts, output_path)
                if os.path.exists(input_video):
                    os.remove(input_video)
            else:
                print("Video already exists, skipping download/cut")
                # Still need info for CSV
                try:
                    info, _ = download_full_video(row.url)
                except Exception as e:
                    info = {"title": row.seq, "uploader": "", "formats":[]}

            # Extract metadata
            meta = ffprobe_metadata(output_path)
            video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
            fps = 30
            width = height = ""
            if video_stream:
                width = video_stream.get("width","")
                height = video_stream.get("height","")
                if "r_frame_rate" in video_stream:
                    num, den = map(int, video_stream["r_frame_rate"].split("/"))
                    fps = num/den if den!=0 else 30

            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            duration = round((end_frame - start_frame)/fps,3)

            # Compute checksum
            checksum = sha256_checksum(output_path)

            # Build row dict
            enriched_row = row.to_dict()
            enriched_row.update({
                "title": info.get("title",""),
                "uploader": info.get("uploader",""),
                "fps": fps,
                "start_time": start_ts,
                "end_time": end_ts,
                "duration": duration,
                "checksum": checksum,
                "width": width,
                "height": height
            })

            # Append row immediately to CSV
            append_row_to_csv(enriched_row, OUTPUT_CSV)

        except Exception as e:
            print(f"❌ Row {idx} failed: {e}")

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Entry Point -------------------

if __name__ == "__main__":
    main()



▶ Processing row 0
[youtube] Extracting URL: https://www.youtube.com/watch?v=B5hrxKe2nP8
[youtube] B5hrxKe2nP8: Downloading webpage
[youtube] B5hrxKe2nP8: Downloading tv client config
[youtube] B5hrxKe2nP8: Downloading player 50cc0679-main
[youtube] B5hrxKe2nP8: Downloading tv player API JSON
[youtube] B5hrxKe2nP8: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] B5hrxKe2nP8: Downloading 1 format(s): 136
[download] Destination: ../data/temp_videos/Parkinsonian Gait Video.mp4
[download] 100% of    8.63MiB in 00:00:02 at 3.31MiB/s   


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.4.4.1)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --


▶ Processing row 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=B5hrxKe2nP8
[youtube] B5hrxKe2nP8: Downloading webpage


[out#0/mp4 @ 0x73ec38e40] video:1212KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.569078%
frame=  510 fps=340 q=-1.0 Lsize=    1219KiB time=00:00:16.96 bitrate= 588.5kbits/s speed=11.3x elapsed=0:00:01.50    
[libx264 @ 0x73ec68a80] frame I:3     Avg QP:17.85  size: 67194
[libx264 @ 0x73ec68a80] frame P:129   Avg QP:19.90  size:  5515
[libx264 @ 0x73ec68a80] frame B:378   Avg QP:24.89  size:   866
[libx264 @ 0x73ec68a80] consecutive B-frames:  1.0%  0.4%  0.6% 98.0%
[libx264 @ 0x73ec68a80] mb I  I16..4: 27.5% 46.2% 26.3%
[libx264 @ 0x73ec68a80] mb P  I16..4:  2.2%  2.3%  0.2%  P16..4: 21.9%  4.6%  3.0%  0.0%  0.0%    skip:65.8%
[libx264 @ 0x73ec68a80] mb B  I16..4:  0.1%  0.1%  0.0%  B16..8: 15.2%  0.4%  0.1%  direct: 0.3%  skip:83.8%  L0:41.6% L1:57.5% BI: 0.9%
[libx264 @ 0x73ec68a80] 8x8 transform intra:47.6% inter:79.7%
[libx264 @ 0x73ec68a80] coded y,uvDC,uvAC intra: 40.7% 72.9% 17.4% inter: 2.6% 4.9% 0.0%
[libx264 @ 0x73ec68a80] i16 v,h,dc,

[youtube] B5hrxKe2nP8: Downloading tv client config
[youtube] B5hrxKe2nP8: Downloading player 50cc0679-main
[youtube] B5hrxKe2nP8: Downloading tv player API JSON
[youtube] B5hrxKe2nP8: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] B5hrxKe2nP8: Downloading 1 format(s): 136
[download] Destination: ../data/temp_videos/Parkinsonian Gait Video.mp4
[download] 100% of    8.63MiB in 00:00:02 at 4.01MiB/s   


...

[out#0/mp4 @ 0xb16c08780] video:1840KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.333599%
frame=  448 fps=200 q=-1.0 Lsize=    1847KiB time=00:00:07.43 bitrate=2035.0kbits/s speed=3.33x elapsed=0:00:02.23    
[libx264 @ 0xb16c90a80] frame I:2     Avg QP:17.20  size: 26180
[libx264 @ 0xb16c90a80] frame P:113   Avg QP:19.67  size:  9286
[libx264 @ 0xb16c90a80] frame B:333   Avg QP:22.84  size:  2349
[libx264 @ 0xb16c90a80] consecutive B-frames:  0.9%  0.0%  0.0% 99.1%
[libx264 @ 0xb16c90a80] mb I  I16..4: 40.0% 56.6%  3.4%
[libx264 @ 0xb16c90a80] mb P  I16..4:  7.0% 14.4%  0.1%  P16..4: 18.9%  2.5%  0.9%  0.0%  0.0%    skip:56.2%
[libx264 @ 0xb16c90a80] mb B  I16..4:  0.2%  0.2%  0.0%  B16..8: 17.6%  0.5%  0.0%  direct: 1.5%  skip:80.1%  L0:46.6% L1:52.3% BI: 1.1%
[libx264 @ 0xb16c90a80] 8x8 transform intra:65.0% inter:86.7%
[libx264 @ 0xb16c90a80] coded y,uvDC,uvAC intra: 5.9% 18.0% 1.5% inter: 1.2% 3.8% 0.0%
[libx264 @ 0xb16c90a80] i16 v,h,dc,p:

[youtube] IV_IsstW-gA: Downloading tv client config
[youtube] IV_IsstW-gA: Downloading player 50cc0679-main
[youtube] IV_IsstW-gA: Downloading tv player API JSON
[youtube] IV_IsstW-gA: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] IV_IsstW-gA: Downloading 1 format(s): 303
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: ../data/temp_videos/Chronic Hemiparetic Gait - Case Study 17.webm
[download] 100% of   16.08MiB in 00:00:04 at 4.02MiB/s   


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.4.4.1)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --


✅ Finished. Enriched CSV saved to ../data/csv_logs/merged_summary_enriched.csv


[out#0/mp4 @ 0xaf8c64300] video:1304KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.363784%
frame=  329 fps=183 q=-1.0 Lsize=    1309KiB time=00:00:05.45 bitrate=1967.8kbits/s speed=3.03x elapsed=0:00:01.79    
[libx264 @ 0xaf8c90a80] frame I:2     Avg QP:16.66  size: 22863
[libx264 @ 0xaf8c90a80] frame P:83    Avg QP:19.68  size:  8848
[libx264 @ 0xaf8c90a80] frame B:244   Avg QP:23.26  size:  2274
[libx264 @ 0xaf8c90a80] consecutive B-frames:  0.9%  0.6%  0.0% 98.5%
[libx264 @ 0xaf8c90a80] mb I  I16..4: 38.3% 58.0%  3.7%
[libx264 @ 0xaf8c90a80] mb P  I16..4:  6.9% 14.4%  0.2%  P16..4: 18.3%  2.4%  0.8%  0.0%  0.0%    skip:57.0%
[libx264 @ 0xaf8c90a80] mb B  I16..4:  0.2%  0.1%  0.0%  B16..8: 17.0%  0.5%  0.0%  direct: 1.2%  skip:81.0%  L0:45.5% L1:53.3% BI: 1.1%
[libx264 @ 0xaf8c90a80] 8x8 transform intra:65.3% inter:88.3%
[libx264 @ 0xaf8c90a80] coded y,uvDC,uvAC intra: 6.3% 13.2% 1.2% inter: 1.2% 3.0% 0.0%
[libx264 @ 0xaf8c90a80] i16 v,h,dc,p:

# Ignores duplicates and takes only the right uploader

In [1]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader with incremental CSV enrichment
and uploader filtering. Avoids duplicates on re-runs.
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/csv_logs/merged_summary_test.csv"
OUTPUT_CSV = "../data/csv_logs/merged_summary_enriched.csv"
TEMP_FOLDER = "../data/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/video_snippets"
TARGET_UPLOADER = "Mission Gait"  # Only download/process videos from this uploader
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

def download_full_video(url):
    ydl_opts = {
        "format": "bv*/b",  # best video or best combined
        "outtmpl": os.path.join(TEMP_FOLDER, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "remote_components": ["ejs:github"],
    }
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)

    if info is None:
        raise RuntimeError("yt-dlp returned no info")
    if info.get("vcodec") == "none":
        raise RuntimeError("Audio-only stream — no video available")

    title = safe_name(info["title"])
    ext = info.get("ext")
    input_path = os.path.join(TEMP_FOLDER, f"{title}.{ext}")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Downloaded file missing: {input_path}")

    return info, input_path

def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    if not os.path.exists(output_file):
        subprocess.run([
            "ffmpeg", "-y",
            "-i", input_file,
            "-ss", start_ts,
            "-to", end_ts,
            "-c:v", "libx264",
            "-c:a", "aac",
            output_file
        ], check=True)

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# -------------------- Main ---------------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    # Load already processed rows to avoid duplicates
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_keys = set(
            zip(df_existing["url"], df_existing["start_frame"], df_existing["end_frame"])
        )
    else:
        processed_keys = set()

    # Add enrichment columns if missing
    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    for idx, row in df.iterrows():
        try:
            key = (row["url"], row["start_frame"], row["end_frame"])
            if key in processed_keys:
                print(f"▶ Row {idx} already processed, skipping")
                continue

            print(f"\n▶ Processing row {idx}")

            url = row["url"]
            start_frame = int(row["start_frame"])
            end_frame = int(row["end_frame"])

            output_name = safe_name(f"{row['seq']}_{row['cam_view']}_{row['gait_event']}_{row['dataset']}_{row['gait_pat']}.mp4")
            output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

            # Download video
            info, input_video = download_full_video(url)
            uploader = info.get("uploader","")
            if uploader != TARGET_UPLOADER:
                print(f"Skipping video '{info.get('title','')}' (Uploader: {uploader})")
                continue

            # Cut clip
            fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
            fps = max(fps_list) if fps_list else 30
            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            cut_and_reencode(input_video, start_ts, end_ts, output_path)

            if os.path.exists(input_video):
                os.remove(input_video)

            # Extract metadata
            meta = ffprobe_metadata(output_path)
            video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
            width = height = ""
            if video_stream:
                width = video_stream.get("width","")
                height = video_stream.get("height","")
                if "r_frame_rate" in video_stream:
                    num, den = map(int, video_stream["r_frame_rate"].split("/"))
                    fps = num/den if den!=0 else 30

            start_ts = frame_to_timestamp(start_frame, fps)
            end_ts = frame_to_timestamp(end_frame, fps)
            duration = round((end_frame - start_frame)/fps,3)
            checksum = sha256_checksum(output_path)

            # Build row dict
            enriched_row = row.to_dict()
            enriched_row.update({
                "title": info.get("title",""),
                "uploader": uploader,
                "fps": fps,
                "start_time": start_ts,
                "end_time": end_ts,
                "duration": duration,
                "checksum": checksum,
                "width": width,
                "height": height
            })

            append_row_to_csv(enriched_row, OUTPUT_CSV)
            processed_keys.add(key)

            print(f"✅ Saved clip: {output_path}")

        except Exception as e:
            print(f"❌ Row {idx} failed: {e}")

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Entry Point -------------------

if __name__ == "__main__":
    main()



▶ Processing row 0
[youtube] Extracting URL: https://www.youtube.com/watch?v=B5hrxKe2nP8
[youtube] B5hrxKe2nP8: Downloading webpage
[youtube] B5hrxKe2nP8: Downloading tv client config
[youtube] B5hrxKe2nP8: Downloading player 50cc0679-main
[youtube] B5hrxKe2nP8: Downloading tv player API JSON
[youtube] B5hrxKe2nP8: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] B5hrxKe2nP8: Downloading 1 format(s): 136
[download] Destination: ../data/temp_videos/Parkinsonian Gait Video.mp4
[download] 100% of    8.63MiB in 00:00:00 at 9.90MiB/s   
Skipping video 'Parkinsonian Gait Video' (Uploader: Carroll College)

▶ Processing row 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=B5hrxKe2nP8
[youtube] B5hrxKe2nP8: Downloading webpage
[youtube] B5hrxKe2nP8: Downloading tv client config
[youtube] B5hrxKe2nP8: Downloading player 50cc0679-main
[youtube] B5hrxKe2nP8: Downloading tv player API JSON
[youtube] B5hrxKe2nP8: Downloading androi

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.4.4.1)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --

✅ Saved clip: ../data/video_snippets/cljawsyn6001o3n6l6z20teaj_right side_nan_Abnormal Gait_abnormal.mp4

▶ Processing row 3
[youtube] Extracting URL: https://www.youtube.com/watch?v=IV_IsstW-gA
[youtube] IV_IsstW-gA: Downloading webpage


[out#0/mp4 @ 0xa7ac40300] video:1840KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.333599%
frame=  448 fps=190 q=-1.0 Lsize=    1847KiB time=00:00:07.43 bitrate=2035.0kbits/s speed=3.15x elapsed=0:00:02.35    
[libx264 @ 0xa7ac6ca80] frame I:2     Avg QP:17.20  size: 26180
[libx264 @ 0xa7ac6ca80] frame P:113   Avg QP:19.67  size:  9286
[libx264 @ 0xa7ac6ca80] frame B:333   Avg QP:22.84  size:  2349
[libx264 @ 0xa7ac6ca80] consecutive B-frames:  0.9%  0.0%  0.0% 99.1%
[libx264 @ 0xa7ac6ca80] mb I  I16..4: 40.0% 56.6%  3.4%
[libx264 @ 0xa7ac6ca80] mb P  I16..4:  7.0% 14.4%  0.1%  P16..4: 18.9%  2.5%  0.9%  0.0%  0.0%    skip:56.2%
[libx264 @ 0xa7ac6ca80] mb B  I16..4:  0.2%  0.2%  0.0%  B16..8: 17.6%  0.5%  0.0%  direct: 1.5%  skip:80.1%  L0:46.6% L1:52.3% BI: 1.1%
[libx264 @ 0xa7ac6ca80] 8x8 transform intra:65.0% inter:86.7%
[libx264 @ 0xa7ac6ca80] coded y,uvDC,uvAC intra: 5.9% 18.0% 1.5% inter: 1.2% 3.8% 0.0%
[libx264 @ 0xa7ac6ca80] i16 v,h,dc,p:

[youtube] IV_IsstW-gA: Downloading tv client config
[youtube] IV_IsstW-gA: Downloading player 50cc0679-main
[youtube] IV_IsstW-gA: Downloading tv player API JSON
[youtube] IV_IsstW-gA: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] IV_IsstW-gA: Downloading 1 format(s): 303
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: ../data/temp_videos/Chronic Hemiparetic Gait - Case Study 17.webm
[download] 100% of   16.08MiB in 00:00:01 at 9.58MiB/s     


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.4.4.1)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --

✅ Saved clip: ../data/video_snippets/cljawu5xd001s3n6lejw8p0uv_left side_nan_Abnormal Gait_abnormal.mp4

✅ Finished. Enriched CSV saved to ../data/csv_logs/merged_summary_enriched.csv


[out#0/mp4 @ 0x76b035c80] video:1304KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.363784%
frame=  329 fps=171 q=-1.0 Lsize=    1309KiB time=00:00:05.45 bitrate=1967.8kbits/s speed=2.84x elapsed=0:00:01.91    
[libx264 @ 0x76b424a80] frame I:2     Avg QP:16.66  size: 22863
[libx264 @ 0x76b424a80] frame P:83    Avg QP:19.68  size:  8848
[libx264 @ 0x76b424a80] frame B:244   Avg QP:23.26  size:  2274
[libx264 @ 0x76b424a80] consecutive B-frames:  0.9%  0.6%  0.0% 98.5%
[libx264 @ 0x76b424a80] mb I  I16..4: 38.3% 58.0%  3.7%
[libx264 @ 0x76b424a80] mb P  I16..4:  6.9% 14.4%  0.2%  P16..4: 18.3%  2.4%  0.8%  0.0%  0.0%    skip:57.0%
[libx264 @ 0x76b424a80] mb B  I16..4:  0.2%  0.1%  0.0%  B16..8: 17.0%  0.5%  0.0%  direct: 1.2%  skip:81.0%  L0:45.5% L1:53.3% BI: 1.1%
[libx264 @ 0x76b424a80] 8x8 transform intra:65.3% inter:88.3%
[libx264 @ 0x76b424a80] coded y,uvDC,uvAC intra: 6.3% 13.2% 1.2% inter: 1.2% 3.0% 0.0%
[libx264 @ 0x76b424a80] i16 v,h,dc,p:

# Final with multiple cores and no download of other videos than mission gate

In [2]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader - Jupyter-safe
Processes every CSV row independently.
Multiple snippets from the same video are allowed.
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing
import threading

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/GAVD_data/csv_logs/merged_summary.csv"
OUTPUT_CSV = "../data/GAVD_data/MissionGate/merged_summary_enriched.csv"
TEMP_FOLDER = "../data/GAVD_data/MissionGate/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/GAVD_data/MissionGate/video_snippets"
TARGET_UPLOADER = "Mission Gait"
MAX_WORKERS = 4
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries",
        "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

# -------------------- yt-dlp helpers ----------------------

def get_video_info_only(url):
    ydl_opts = {"quiet": True, "skip_download": True, "noplaylist": True}
    with YoutubeDL(ydl_opts) as ydl:
        return ydl.extract_info(url, download=False)

def download_full_video(url, lock, video_cache):
    """Download full video if not already cached"""
    with lock:
        if url in video_cache:
            return video_cache[url]

    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(TEMP_FOLDER, "%(id)s.%(ext)s"),
        "noplaylist": True,
        "quiet": False,
        "cachedir": False,
        "allow_unplayable_formats": True,
    }

    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
    except Exception as e:
        print(f"⚠️  yt-dlp failed on {url}: {e}, trying fallback to 'best'")
        ydl_opts["format"] = "best"
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)

    input_path = ydl.prepare_filename(info)

    if not os.path.exists(input_path):
        # Fallback: pick any mp4 in TEMP_FOLDER starting with video id
        vid_id = info.get("id")
        candidates = [f for f in os.listdir(TEMP_FOLDER) if f.startswith(vid_id) and f.endswith(".mp4")]
        if not candidates:
            raise FileNotFoundError(f"Downloaded file missing for {url}")
        input_path = os.path.join(TEMP_FOLDER, candidates[0])

    with lock:
        video_cache[url] = (input_path, info)

    return input_path, info

def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    subprocess.run(
        ["ffmpeg", "-y", "-i", input_file, "-ss", start_ts, "-to", end_ts,
         "-c:v", "libx264", "-c:a", "aac", output_file],
        check=True
    )

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# -------------------- Worker ----------------------

def process_row(idx, row, video_cache, lock):
    """Process a single CSV row independently"""
    url = row["url"]
    try:
        info = get_video_info_only(url)
        uploader = info.get("uploader", "")
        if uploader != TARGET_UPLOADER:
            print(f"⏭ Skipping row {idx} (Uploader '{uploader}')")
            return None

        input_video, info = download_full_video(url, lock, video_cache)

        start_frame = int(row["start_frame"])
        end_frame = int(row["end_frame"])
        fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
        fps = max(fps_list) if fps_list else 30

        start_ts = frame_to_timestamp(start_frame, fps)
        end_ts = frame_to_timestamp(end_frame, fps)

        output_name = safe_name(
            f"{row['seq']}_{row['cam_view']}_{row['gait_event']}_"
            f"{row['dataset']}_{row['gait_pat']}_frames{start_frame}-{end_frame}.mp4"
        )
        output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

        cut_and_reencode(input_video, start_ts, end_ts, output_path)

        meta = ffprobe_metadata(output_path)
        video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
        width = video_stream.get("width", "") if video_stream else ""
        height = video_stream.get("height", "") if video_stream else ""
        duration = round((end_frame - start_frame)/fps, 3)
        checksum = sha256_checksum(output_path)

        enriched_row = row.to_dict()
        enriched_row.update({
            "title": info.get("title",""),
            "uploader": uploader,
            "fps": fps,
            "start_time": start_ts,
            "end_time": end_ts,
            "duration": duration,
            "checksum": checksum,
            "width": width,
            "height": height,
        })

        return enriched_row, output_path

    except Exception as e:
        print(f"❌ Row {idx} failed: {e}")
        return None

# -------------------- Main ----------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    workers = min(MAX_WORKERS, max(1, multiprocessing.cpu_count() - 1))
    print(f"▶ Running with {workers} workers (Jupyter-safe threads)")

    video_cache = {}
    lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_row, idx, row, video_cache, lock): idx
                   for idx, row in df.iterrows()}

        for future in as_completed(futures):
            idx = futures[future]
            result = future.result()
            if result is None:
                continue
            enriched_row, output_path = result
            append_row_to_csv(enriched_row, OUTPUT_CSV)
            print(f"✅ Saved clip: {output_path}")

    # Cleanup cached videos
    for input_video, _ in video_cache.values():
        if os.path.exists(input_video):
            os.remove(input_video)

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Run ----------------------
main()


▶ Running with 4 workers (Jupyter-safe threads)


         If you experience any issues while using this option, DO NOT open a bug report


[youtube] Extracting URL: https://www.youtube.com/watch?v=TgkxrrhnvlM
[youtube] TgkxrrhnvlM: Downloading webpage
⏭ Skipping row 1 (Uploader 'Carroll College')
⏭ Skipping row 0 (Uploader 'Carroll College')


         If you experience any issues while using this option, DO NOT open a bug report


[youtube] Extracting URL: https://www.youtube.com/watch?v=TgkxrrhnvlM
[youtube] TgkxrrhnvlM: Downloading webpage
[youtube] TgkxrrhnvlM: Downloading tv client config
[youtube] TgkxrrhnvlM: Downloading player 50cc0679-main
[youtube] TgkxrrhnvlM: Downloading tv client config
[youtube] TgkxrrhnvlM: Downloading tv player API JSON
[youtube] TgkxrrhnvlM: Downloading player c80790c5-main
[youtube] TgkxrrhnvlM: Downloading android sdkless player API JSON
[youtube] TgkxrrhnvlM: Downloading tv player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno


...

❌ Row 1877 failed: ERROR: [youtube] FFki8FtaByw: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
❌ Row 1878 failed: ERROR: [youtube] FFki8FtaByw: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


❌ Row 1879 failed: ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


❌ Row 1880 failed: ERROR: [youtube] Ha9LKXZfWBQ: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

✅ Finished. Enriched CSV saved to ../data/GAVD_data/MissionGate/merged_summary_enriched.csv


# Latest version with hopefully working videos - not tested yet

In [ ]:
#!/usr/bin/env python3
"""
YouTube Segment Downloader - Jupyter-safe
Processes every CSV row independently.
Multiple snippets from the same video are allowed.
Includes robust checks to avoid broken snippets.
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import pandas as pd
import re
import hashlib
import json
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing
import threading
import time

# -------------------- USER SETTINGS --------------------
INPUT_CSV = "../data/GAVD_data/csv_logs/merged_summary.csv"
OUTPUT_CSV = "../data/GAVD_data/MissionGate/merged_summary_enriched.csv"
TEMP_FOLDER = "../data/GAVD_data/MissionGate/temp_videos"
OUTPUT_VIDEO_FOLDER = "../data/GAVD_data/MissionGate/video_snippets"
TARGET_UPLOADER = "Mission Gait"
MAX_WORKERS = 4
# ------------------------------------------------------

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_VIDEO_FOLDER, exist_ok=True)

# -------------------- Utilities ----------------------

def safe_name(s):
    return re.sub(r"[^\w\-_. ]", "_", str(s)).strip()

def frame_to_timestamp(frame, fps):
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def sha256_checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_metadata(file_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries",
        "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate",
        "-print_format", "json",
        file_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed on {file_path}")
    return json.loads(result.stdout)

def verify_video_file(path):
    """Check if video exists and has valid duration > 0"""
    if not os.path.exists(path):
        return False
    try:
        meta = ffprobe_metadata(path)
        duration = float(meta["format"]["duration"])
        return duration > 0
    except Exception:
        return False

def append_row_to_csv(row_dict, csv_file):
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

# -------------------- yt-dlp helpers ----------------------

def get_video_info_only(url):
    ydl_opts = {"quiet": True, "skip_download": True, "noplaylist": True}
    with YoutubeDL(ydl_opts) as ydl:
        return ydl.extract_info(url, download=False)

def download_full_video(url, lock, video_cache, retries=2):
    """Download full video if not already cached, with retry and verification"""
    with lock:
        if url in video_cache:
            return video_cache[url]

    for attempt in range(retries):
        try:
            ydl_opts = {
                "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]",
                "merge_output_format": "mp4",
                "outtmpl": os.path.join(TEMP_FOLDER, "%(id)s.%(ext)s"),
                "noplaylist": True,
                "quiet": False,
                "cachedir": False,
                "remote_components": ["ejs:github"],
            }

            with YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                input_path = ydl.prepare_filename(info)

            if verify_video_file(input_path):
                with lock:
                    video_cache[url] = (input_path, info)
                return input_path, info
            else:
                print(f"⚠ Downloaded video invalid, retrying ({attempt+1}/{retries})")
                time.sleep(2)
        except Exception as e:
            print(f"⚠ Download attempt {attempt+1} failed: {e}")
            time.sleep(2)

    # Fallback if still fails
    raise RuntimeError(f"Failed to download valid video after {retries} attempts: {url}")

def cut_and_reencode(input_file, start_ts, end_ts, output_file):
    subprocess.run(
        ["ffmpeg", "-y", "-i", input_file, "-ss", start_ts, "-to", end_ts,
         "-c:v", "libx264", "-c:a", "aac", output_file],
        check=True
    )
    if not verify_video_file(output_file):
        raise RuntimeError(f"Snippet is invalid or zero-length: {output_file}")

# -------------------- Worker ----------------------

def process_row(idx, row, video_cache, lock):
    url = row["url"]
    try:
        info = get_video_info_only(url)
        uploader = info.get("uploader", "")
        if uploader != TARGET_UPLOADER:
            print(f"⏭ Skipping row {idx} (Uploader '{uploader}')")
            return None

        input_video, info = download_full_video(url, lock, video_cache)

        start_frame = int(row["start_frame"])
        end_frame = int(row["end_frame"])
        fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
        fps = max(fps_list) if fps_list else 30

        start_ts = frame_to_timestamp(start_frame, fps)
        end_ts = frame_to_timestamp(end_frame, fps)

        output_name = safe_name(
            f"{row['seq']}_{row['cam_view']}_{row['gait_event']}_"
            f"{row['dataset']}_{row['gait_pat']}_frames{start_frame}-{end_frame}.mp4"
        )
        output_path = os.path.join(OUTPUT_VIDEO_FOLDER, output_name)

        cut_and_reencode(input_video, start_ts, end_ts, output_path)

        meta = ffprobe_metadata(output_path)
        video_stream = next((s for s in meta["streams"] if s["codec_type"]=="video"), None)
        width = video_stream.get("width", "") if video_stream else ""
        height = video_stream.get("height", "") if video_stream else ""
        duration = round((end_frame - start_frame)/fps, 3)
        checksum = sha256_checksum(output_path)

        enriched_row = row.to_dict()
        enriched_row.update({
            "title": info.get("title",""),
            "uploader": uploader,
            "fps": fps,
            "start_time": start_ts,
            "end_time": end_ts,
            "duration": duration,
            "checksum": checksum,
            "width": width,
            "height": height,
        })

        return enriched_row, output_path

    except Exception as e:
        print(f"❌ Row {idx} failed: {e}")
        return None

# -------------------- Main ----------------------

def main():
    df = pd.read_csv(INPUT_CSV)

    # Ensure enrichment columns exist
    for col in ["title","uploader","fps","start_time","end_time","duration","checksum","width","height"]:
        if col not in df.columns:
            df[col] = ""

    workers = min(MAX_WORKERS, max(1, multiprocessing.cpu_count() - 1))
    print(f"▶ Running with {workers} workers (Jupyter-safe threads)")

    video_cache = {}
    lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_row, idx, row, video_cache, lock): idx
                   for idx, row in df.iterrows()}

        for future in as_completed(futures):
            idx = futures[future]
            result = future.result()
            if result is None:
                continue
            enriched_row, output_path = result
            append_row_to_csv(enriched_row, OUTPUT_CSV)
            print(f"✅ Saved clip: {output_path}")

    # Cleanup cached videos
    for input_video, _ in video_cache.values():
        if os.path.exists(input_video):
            os.remove(input_video)

    print(f"\n✅ Finished. Enriched CSV saved to {OUTPUT_CSV}")

# -------------------- Run ----------------------
if __name__ == "__main__":
    main()


## Running scripts for video and csv extraction only - manual input

In [20]:
#!/usr/bin/env python3
"""
YouTube QuickTime-Compatible Segment Downloader by Frame Number
Requirements:
- Python 3
- yt-dlp (`pip install yt-dlp`)
- ffmpeg installed and in PATH
- Optional: Deno installed for JS challenges
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import sys

# -------------------- USER SETTINGS --------------------
URLS_FILE = "../data/videourls.txt"         # Text file with YouTube URLs
TARGET_UPLOADER = "Mission Gait"            # Only download videos from this uploader
START_FRAME = 1000                           # Start frame
END_FRAME = 1300                             # End frame
OUTPUT_TEMPLATE = "%(title)s_%(section_start)s-%(section_end)s.mp4"
TEMP_FOLDER = "temp_videos"                  # Temporary folder for full downloads
# -------------------------------------------------------

def frame_to_timestamp(frame, fps):
    """Convert frame number to HH:MM:SS.sss timestamp."""
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def ensure_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)

def download_full_video(url, temp_folder):
    """Download best MP4 video + audio, merged into MP4. Skip if unavailable."""
    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(temp_folder, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "ignoreerrors": True,
        "no_warnings": True,
        "quiet": False,
        "remote_components": "ejs:github",  # solves JS challenges
    }
    with YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=True)
            if info is None:
                print(f"Skipping {url} — no suitable video format available")
                return None
            return info
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            return None

def cut_video_segment(input_file, start_ts, end_ts, output_file):
    """Cut a segment and re-encode to QuickTime-compatible MP4 (H.264 + AAC)."""
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_file,
        "-ss", start_ts,
        "-to", end_ts,
        "-c:v", "libx264",
        "-c:a", "aac",
        "-strict", "experimental",
        output_file
    ]
    subprocess.run(cmd, check=True)

def process_video(url, start_frame, end_frame, target_uploader, output_template, temp_folder):
    info = download_full_video(url, temp_folder)
    if info is None:
        return  # Skip this video

    uploader = info.get("uploader")
    if uploader != target_uploader:
        print(f"Skipping '{info.get('title', 'Unknown')}' (Uploader: {uploader})")
        return

    # Determine FPS safely
    fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
    fps = max(fps_list) if fps_list else 30
    print(f"Processing '{info.get('title', 'Unknown')}' (FPS: {fps}, Uploader: {uploader})")

    # Convert frames to timestamps
    start_ts = frame_to_timestamp(start_frame, fps)
    end_ts = frame_to_timestamp(end_frame, fps)

    # Determine downloaded file path (MP4)
    input_file = os.path.join(temp_folder, f"{info['title']}.mp4")
    if not os.path.exists(input_file):
        # Try original extension if MP4 not available
        ext = info.get("ext") or "mp4"
        input_file = os.path.join(temp_folder, f"{info['title']}.{ext}")
        if not os.path.exists(input_file):
            print(f"Skipping '{info['title']}' — video file not found")
            return

    # Build output filename
    output_file = output_template.replace("%(title)s", info['title'])\
                                 .replace("%(section_start)s", start_ts)\
                                 .replace("%(section_end)s", end_ts)

    # Cut segment and re-encode to QuickTime-compatible MP4
    cut_video_segment(input_file, start_ts, end_ts, output_file)
    print(f"Saved segment: {output_file}")

    # Delete temp full video
    if os.path.exists(input_file):
        os.remove(input_file)

def main():
    ensure_folder(TEMP_FOLDER)

    # Load URLs
    try:
        with open(URLS_FILE, "r") as f:
            urls = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Error: File '{URLS_FILE}' not found.")
        sys.exit(1)

    # Process each video
    for url in urls:
        try:
            process_video(url, START_FRAME, END_FRAME, TARGET_UPLOADER, OUTPUT_TEMPLATE, TEMP_FOLDER)
        except Exception as e:
            print(f"Error processing {url}: {e}")

if __name__ == "__main__":
    main()


[youtube] Extracting URL: https://www.youtube.com/watch?v=IV_IsstW-gA
[youtube] IV_IsstW-gA: Downloading webpage
[youtube] IV_IsstW-gA: Downloading tv client config
[youtube] IV_IsstW-gA: Downloading player 50cc0679-main
[youtube] IV_IsstW-gA: Downloading tv player API JSON
[youtube] IV_IsstW-gA: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] IV_IsstW-gA: Downloading 1 format(s): 139
[download] Destination: temp_videos/Chronic Hemiparetic Gait - Case Study 17.m4a
[download] 100% of    1.20MiB in 00:00:00 at 1.25MiB/s   
[FixupM4a] Correcting container of "temp_videos/Chronic Hemiparetic Gait - Case Study 17.m4a"
Processing 'Chronic Hemiparetic Gait - Case Study 17' (FPS: 0.5072463768115942, Uploader: Mission Gait)
Saved segment: Chronic Hemiparetic Gait - Case Study 17_00:32:51.429-00:42:42.857.mp4


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.4.4.1)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enable-libsoxr --

In [21]:
#!/usr/bin/env python3
"""
YouTube QuickTime-Compatible Segment Downloader by Frame Number
With CSV logging of video info.

Requirements:
- Python 3
- yt-dlp (`pip install yt-dlp`)
- ffmpeg installed and in PATH
- Optional: Deno installed for JS challenges
"""

from yt_dlp import YoutubeDL
import os
import subprocess
import sys
import csv

# -------------------- USER SETTINGS --------------------
URLS_FILE = "../data/videourls.txt"         # Text file with YouTube URLs
TARGET_UPLOADER = "Mission Gait"            # Only download videos from this uploader
START_FRAME = 1000                           # Start frame
END_FRAME = 1300                             # End frame
OUTPUT_TEMPLATE = "%(title)s_%(section_start)s-%(section_end)s.mp4"
TEMP_FOLDER = "temp_videos"                  # Temporary folder for full downloads
CSV_LOG_FILE = "video_log.csv"               # CSV file to save segment info
# -------------------------------------------------------

def frame_to_timestamp(frame, fps):
    """Convert frame number to HH:MM:SS.sss timestamp."""
    seconds = frame / fps
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02}:{m:02}:{s:06.3f}"

def ensure_folder(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)

def download_full_video(url, temp_folder):
    """Download best MP4 video + audio, merged into MP4. Skip if unavailable."""
    ydl_opts = {
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": os.path.join(temp_folder, "%(title)s.%(ext)s"),
        "noplaylist": True,
        "ignoreerrors": True,
        "no_warnings": True,
        "quiet": False,
        "remote_components": "ejs:github",  # solves JS challenges
    }
    with YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=True)
            if info is None:
                print(f"Skipping {url} — no suitable video format available")
                return None
            return info
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            return None

def cut_video_segment(input_file, start_ts, end_ts, output_file):
    """Cut a segment and re-encode to QuickTime-compatible MP4 (H.264 + AAC)."""
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")
    
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_file,
        "-ss", start_ts,
        "-to", end_ts,
        "-c:v", "libx264",
        "-c:a", "aac",
        "-strict", "experimental",
        output_file
    ]
    subprocess.run(cmd, check=True)

def log_to_csv(row, csv_file):
    """Append a row to CSV, create file if it doesn't exist."""
    file_exists = os.path.exists(csv_file)
    with open(csv_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def process_video(url, start_frame, end_frame, target_uploader, output_template, temp_folder, csv_file):
    info = download_full_video(url, temp_folder)
    if info is None:
        return  # Skip this video

    uploader = info.get("uploader")
    if uploader != target_uploader:
        print(f"Skipping '{info.get('title', 'Unknown')}' (Uploader: {uploader})")
        return

    # Determine FPS safely
    fps_list = [f.get("fps") for f in info.get("formats", []) if f.get("fps")]
    fps = max(fps_list) if fps_list else 30
    print(f"Processing '{info.get('title', 'Unknown')}' (FPS: {fps}, Uploader: {uploader})")

    # Convert frames to timestamps
    start_ts = frame_to_timestamp(start_frame, fps)
    end_ts = frame_to_timestamp(end_frame, fps)
    duration = round((end_frame - start_frame) / fps, 3)

    # Determine downloaded file path (MP4)
    input_file = os.path.join(temp_folder, f"{info['title']}.mp4")
    if not os.path.exists(input_file):
        ext = info.get("ext") or "mp4"
        input_file = os.path.join(temp_folder, f"{info['title']}.{ext}")
        if not os.path.exists(input_file):
            print(f"Skipping '{info['title']}' — video file not found")
            return

    # Build output filename
    output_file = output_template.replace("%(title)s", info['title'])\
                                 .replace("%(section_start)s", start_ts)\
                                 .replace("%(section_end)s", end_ts)

    # Cut segment and re-encode to QuickTime-compatible MP4
    cut_video_segment(input_file, start_ts, end_ts, output_file)
    print(f"Saved segment: {output_file}")

    # Delete temp full video
    if os.path.exists(input_file):
        os.remove(input_file)

    # Log info to CSV
    row = {
        "title": info['title'],
        "url": url,
        "uploader": uploader,
        "fps": fps,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "start_time": start_ts,
        "end_time": end_ts,
        "duration": duration
    }
    log_to_csv(row, csv_file)

def main():
    ensure_folder(TEMP_FOLDER)

    # Load URLs
    try:
        with open(URLS_FILE, "r") as f:
            urls = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Error: File '{URLS_FILE}' not found.")
        sys.exit(1)

    # Process each video
    for url in urls:
        try:
            process_video(url, START_FRAME, END_FRAME, TARGET_UPLOADER, OUTPUT_TEMPLATE, TEMP_FOLDER, CSV_LOG_FILE)
        except Exception as e:
            print(f"Error processing {url}: {e}")

if __name__ == "__main__":
    main()


[youtube] Extracting URL: https://www.youtube.com/watch?v=IV_IsstW-gA
[youtube] IV_IsstW-gA: Downloading webpage
[youtube] IV_IsstW-gA: Downloading tv client config
[youtube] IV_IsstW-gA: Downloading player 50cc0679-main
[youtube] IV_IsstW-gA: Downloading tv player API JSON
[youtube] IV_IsstW-gA: Downloading android sdkless player API JSON
[youtube] [jsc:deno] Solving JS challenges using deno
[info] IV_IsstW-gA: Downloading 1 format(s): 139


ERROR: unable to download video data: HTTP Error 403: Forbidden


Processing 'Chronic Hemiparetic Gait - Case Study 17' (FPS: 0.5072463768115942, Uploader: Mission Gait)
Skipping 'Chronic Hemiparetic Gait - Case Study 17' — video file not found
